# Differential Diagnosis: PCOS vs Endometriosis vs Healthy

This notebook demonstrates a beginner-friendly approach to building a clinically interpretable
differential diagnosis system. It synthesizes a small dataset, trains a classifier, shows a
confusion matrix, produces SHAP explainability plots, and outputs clinical reasoning summaries.

In [ ]:
# Install dependencies if running in Colab
# (Uncomment the following lines in a fresh Colab runtime)
# !pip install scikit-learn shap matplotlib seaborn pandas

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# For reproducibility
np.random.seed(42)

In [ ]:
# Synthetic dataset: features chosen to reflect clinical signals
N = 1000
data = {
    'amh': np.concatenate([np.random.normal(6,1, int(N*0.4)), np.random.normal(2,0.8, int(N*0.4)), np.random.normal(1.5,0.5, int(N*0.2))]),
    'cycleLength': np.concatenate([np.random.normal(40,5, int(N*0.4)), np.random.normal(30,3, int(N*0.4)), np.random.normal(28,2, int(N*0.2))]),
    'pelvicPain': np.concatenate([np.random.binomial(1,0.2, int(N*0.4)), np.random.binomial(1,0.6, int(N*0.4)), np.random.binomial(1,0.05, int(N*0.2))]),
    'follicleCountMax': np.concatenate([np.random.normal(14,2, int(N*0.4)), np.random.normal(13,1.5, int(N*0.4)), np.random.normal(8,1, int(N*0.2))]),
}
# Labels: 0=PCOS,1=Endometriosis,2=Healthy (ordered to match requested UI)
labels = np.concatenate([np.zeros(int(N*0.4)), np.ones(int(N*0.4)), np.full(int(N*0.2),2)])
df = pd.DataFrame(data)
df['label'] = labels.astype(int)
df['pelvicPain'] = df['pelvicPain'].astype(int)
df = df.sample(frac=1).reset_index(drop=True)
df.head()

In [ ]:
# Train/test split and model training
X = df[['amh','cycleLength','pelvicPain','follicleCountMax']]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['PCOS','Endometriosis','Healthy']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['PCOS','Endometriosis','Healthy'], yticklabels=['PCOS','Endometriosis','Healthy'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# SHAP explainability (small sample for speed)
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test.iloc[:200])
plt.title('SHAP summary (PCOS class)')
shap.summary_plot(shap_values[0], X_test.iloc[:200], show=False)
plt.show()

### Clinical reasoning summaries
The model uses elevated AMH, prolonged cycle length, and increased follicle counts to distinguish PCOS from Endometriosis. Endometriosis is driven by pelvic pain and dysmenorrhea signals in our synthetic set. Healthy controls lack these features.